# Pairing Results Computation

This notebook runs the selected pairing-model PMM, compression, and neural-network baseline computations. It saves raw arrays, summary tables, and run settings only. It does not create plots.

This notebook was adapted from a larger original notebook written by myself, with PMM code mostly based on code from Morten. Codex and ChatGPT helped organize the pairing experiments and reuse saved computations instead of recomputing repeated quantities.


In [4]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd

from common import (
    GLOBAL_SEED,
    DEFAULT_EPOCHS,
    DEFAULT_LR,
    DEFAULT_INIT_SCALE,
    DEFAULT_Q_GRID,
    DEFAULT_ERR_GRID,
    DEFAULT_R_STEP,
    DEFAULT_REFINE_LOCAL,
    compute_error_stats,
    test_error_summary,
)
from data import make_pairing_data, make_train_validation_split
from pmm import pmm_parameter_count, compressed_pmm_predictions, train_eigenvalue_pmm_on_data as train_pmm_on_data
from compression import prepare_pmm_compression
from benchmarks import pmm_runtime_row as runtime_row
from baselines import run_mlp_grid_search, train_final_mlp, evaluate_mlp_on_data

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
PAIRING_RESULTS_PATH = RESULTS_DIR / "pairing_results.pkl"

PMM_SWEEP_EPOCHS = 20000
PAIRING_PMM_SIZES = [16, 32, 48]
PAIRING_SCALING_EPOCH_BUDGETS = [5000, 10000, 15000, 20000]
NN_EPOCHS = 2000


def save_results(results, path=PAIRING_RESULTS_PATH):
    """Save the current result dict to a pickle path and return None."""
    with path.open("wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved pairing results to {path}")


def compact_run_for_save(case, compression, data, regime):
    """Collect representative pairing arrays and metadata for saving and plotting.

    Codex and ChatGPT helped separate this compact save format from the
    plotting notebook so results could be reused directly.
    """
    compressed_pred = compressed_pmm_predictions(
        case["M0"],
        case["M1"],
        compression["B_rstar"],
        data["X_test"],
        int(case["k"]),
    )
    return {
        "metadata": {
            "regime": regime,
            "n": int(case["n_pmm"]),
            "k": int(case["k"]),
            "seed": int(case["seed"]),
            "epochs": int(case["epochs"]),
            "r_star": int(compression["r_star"]),
            "r_star_over_k": float(compression["r_star_over_k"]),
            "e_rstar": float(compression["e_rstar"]),
        },
        "g_test": np.asarray(data["g_test"]),
        "X_test": np.asarray(data["X_test"]),
        "exact": np.asarray(data["E_test"]),
        "pmm": np.asarray(case["pmm_pred"]),
        "compressed_pmm": compressed_pred,
        "pmm_abs_error": np.abs(np.asarray(case["pmm_pred"]) - np.asarray(data["E_test"])),
        "compressed_abs_error": np.abs(compressed_pred - np.asarray(data["E_test"])),
    }


pairing_results = {}


## 1. Pairing Regime Sweep

Settings: `pnum = hnum = 10`, `k = 3`, tolerance `1e-2`, intervals `[0, 2]` and `[0, 4]`, and PMM sizes `n in {16, 32, 48}`.


In [5]:
pairing_sweep_settings = {
    "pnum": 10,
    "hnum": 10,
    "delta": 1.0,
    "k": 3,
    "seeds": [GLOBAL_SEED, GLOBAL_SEED + 1, GLOBAL_SEED + 2],
    "tol": 1e-2,
    "n_pmm_values": PAIRING_PMM_SIZES,
    "intervals": [
        {"name": "g0_2", "g_min": 0.0, "g_max": 2.0, "n_train": 40, "n_test": 200},
        {"name": "g0_4", "g_min": 0.0, "g_max": 4.0, "n_train": 40, "n_test": 200},
    ],
    "epochs": PMM_SWEEP_EPOCHS,
    "learning_rate": DEFAULT_LR,
    "init_scale": DEFAULT_INIT_SCALE,
    "q_grid": DEFAULT_Q_GRID,
    "err_grid": DEFAULT_ERR_GRID,
    "r_step": DEFAULT_R_STEP,
    "refine": DEFAULT_REFINE_LOCAL,
    "representative_selection_metric": "compressed_test_mean_abs_error",
    "method_notes": {
        "representative_selection": "Representative runs are selected by compressed PMM test mean absolute error.",
        "runtime_table": "Runtime columns benchmark learned PMM scans and compressed learned-PMM scans, not exact many-body solves.",
    },
}

sweep_rows = []
runtime_rows = []
run_lookup = {}
representatives = {}

for interval in pairing_sweep_settings["intervals"]:
    regime_name = interval["name"]
    data = make_pairing_data(
        pnum=pairing_sweep_settings["pnum"],
        hnum=pairing_sweep_settings["hnum"],
        delta=pairing_sweep_settings["delta"],
        gmin=interval["g_min"],
        gmax=interval["g_max"],
        n_train=interval["n_train"],
        n_test=interval["n_test"],
        k_levels=pairing_sweep_settings["k"],
    )

    for n_pmm in pairing_sweep_settings["n_pmm_values"]:
        for seed in pairing_sweep_settings["seeds"]:
            print(f"Pairing sweep: regime={regime_name}, n={n_pmm}, seed={seed}")
            case = train_pmm_on_data(
                data,
                n_pmm=n_pmm,
                k=pairing_sweep_settings["k"],
                seed=seed,
                epochs=pairing_sweep_settings["epochs"],
                lr=pairing_sweep_settings["learning_rate"],
                init_scale=pairing_sweep_settings["init_scale"],
            )
            comp = prepare_pmm_compression(
                case,
                tol=pairing_sweep_settings["tol"],
                n_grid_Q=pairing_sweep_settings["q_grid"],
                n_grid_err=pairing_sweep_settings["err_grid"],
                r_step=pairing_sweep_settings["r_step"],
                refine=pairing_sweep_settings["refine"],
            )
            compressed_pred = compressed_pmm_predictions(case["M0"], case["M1"], comp["B_rstar"], data["X_test"], pairing_sweep_settings["k"])
            compressed_stats = test_error_summary(data["E_test"], compressed_pred)

            row = {
                "regime": regime_name,
                "g_min": interval["g_min"],
                "g_max": interval["g_max"],
                "pnum": pairing_sweep_settings["pnum"],
                "hnum": pairing_sweep_settings["hnum"],
                "n": int(n_pmm),
                "k": pairing_sweep_settings["k"],
                "seed": int(seed),
                "epochs": int(case["epochs"]),
                "train_loss": case["train_loss"],
                "test_loss": case["test_loss"],
                "test_mean_abs_error": case["test_mean_abs_error"],
                "test_max_abs_error": case["test_max_abs_error"],
                "test_rms_error": case["test_rms_error"],
                "compressed_test_loss": compressed_stats["test_loss"],
                "compressed_test_mean_abs_error": compressed_stats["test_mean_abs_error"],
                "compressed_test_max_abs_error": compressed_stats["test_max_abs_error"],
                "compressed_test_rms_error": compressed_stats["test_rms_error"],
                "train_time_s": case["train_time_s"],
                "r_star": comp["r_star"],
                "r_star_over_k": comp["r_star_over_k"],
                "e_rstar": comp["e_rstar"],
                "e_2k": comp["e_2k"],
                "compression_preprocess_time_s": comp["compression_preprocess_time_s"],
            }
            sweep_rows.append(row)
            runtime_rows.append(runtime_row(case, comp, regime=regime_name, n_deploy=len(data["X_test"])))

            key = (regime_name, int(n_pmm), int(seed))
            run_lookup[key] = compact_run_for_save(case, comp, data, regime=regime_name)

sweep_detail = pd.DataFrame(sweep_rows)
regime_summary = (
    sweep_detail.groupby(["regime", "g_min", "g_max", "n", "epochs"], as_index=False)
    .agg(
        mean_test_loss=("test_loss", "mean"),
        std_test_loss=("test_loss", "std"),
        mean_train_time_s=("train_time_s", "mean"),
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        mean_test_mean_abs_error=("test_mean_abs_error", "mean"),
        mean_compressed_test_mean_abs_error=("compressed_test_mean_abs_error", "mean"),
    )
    .sort_values(["regime", "mean_test_loss", "n"])
    .reset_index(drop=True)
)
regime_summary["std_test_loss"] = regime_summary["std_test_loss"].fillna(0.0)
runtime_detail = pd.DataFrame(runtime_rows)
runtime_summary = (
    runtime_detail.groupby(["regime", "n", "epochs"], as_index=False)
    .agg(
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        pmm_scan_time_s=("pmm_scan_time_s", "mean"),
        pmm_compressed_scan_time_s=("pmm_compressed_scan_time_s", "mean"),
        compression_preprocess_time_s=("compression_preprocess_time_s", "mean"),
        total_reduced_time_s=("total_reduced_time_s", "mean"),
        break_even_scan_count=("break_even_scan_count", "mean"),
        raw_speedup=("raw_speedup", "mean"),
        amortized_speedup=("amortized_speedup", "mean"),
    )
    .sort_values(["regime", "n"])
    .reset_index(drop=True)
)

representative_metric = pairing_sweep_settings["representative_selection_metric"]
for regime_name in sweep_detail["regime"].unique():
    best = sweep_detail[sweep_detail["regime"].eq(regime_name)].sort_values(representative_metric).iloc[0]
    representative = run_lookup[(best["regime"], int(best["n"]), int(best["seed"]))]
    representative["metadata"]["selection_metric"] = representative_metric
    representatives[regime_name] = representative

pairing_results["regime_sweep"] = {
    "settings": pairing_sweep_settings,
    "detail": sweep_detail,
    "summary": regime_summary,
    "runtime_detail": runtime_detail,
    "runtime_summary": runtime_summary,
    "representatives": representatives,
    "runs_by_key": run_lookup,
}
save_results(pairing_results)
regime_summary


Pairing sweep: regime=g0_2, n=16, seed=1234
Pairing sweep: regime=g0_2, n=16, seed=1235
Pairing sweep: regime=g0_2, n=16, seed=1236
Pairing sweep: regime=g0_2, n=32, seed=1234
Pairing sweep: regime=g0_2, n=32, seed=1235
Pairing sweep: regime=g0_2, n=32, seed=1236
Pairing sweep: regime=g0_2, n=48, seed=1234
Pairing sweep: regime=g0_2, n=48, seed=1235
Pairing sweep: regime=g0_2, n=48, seed=1236
Pairing sweep: regime=g0_4, n=16, seed=1234
Pairing sweep: regime=g0_4, n=16, seed=1235
Pairing sweep: regime=g0_4, n=16, seed=1236
Pairing sweep: regime=g0_4, n=32, seed=1234
Pairing sweep: regime=g0_4, n=32, seed=1235
Pairing sweep: regime=g0_4, n=32, seed=1236
Pairing sweep: regime=g0_4, n=48, seed=1234
Pairing sweep: regime=g0_4, n=48, seed=1235
Pairing sweep: regime=g0_4, n=48, seed=1236
Saved pairing results to results/pairing_results.pkl


,regime,g_min,g_max,n,epochs,mean_test_loss,std_test_loss,mean_train_time_s,mean_r_star,mean_r_star_over_k,mean_e_rstar,mean_test_mean_abs_error,mean_compressed_test_mean_abs_error
0,g0_2,0.0,2.0,48,20000,0.000001,4.531481e-07,98.102943,12.333333,4.111111,0.006674,0.000861,0.001286
1,g0_2,0.0,2.0,16,20000,0.000001,3.467122e-07,12.255272,11.666667,3.888889,0.003473,0.000846,0.000979
2,g0_2,0.0,2.0,32,20000,0.000001,7.338968e-07,41.387588,12.666667,4.222222,0.003981,0.000871,0.000973
3,g0_4,0.0,4.0,32,20000,0.000001,8.522669e-08,45.494406,12.666667,4.222222,0.006148,0.000886,0.001208
4,g0_4,0.0,4.0,16,20000,0.000001,1.340890e-07,12.289826,12.666667,4.222222,0.000777,0.000980,0.000975
5,g0_4,0.0,4.0,48,20000,0.000002,2.225793e-07,107.503847,15.333333,5.111111,0.004701,0.001023,0.001155


## 2. Pairing PMM Epoch Scaling

Uses the best pairing regime from the sweep above, then varies the epoch budget for all PMM sizes. For each size it uses the best representative seed selected by compressed PMM test mean absolute error.


In [6]:
pairing_scaling_selection_metric = "mean_compressed_test_mean_abs_error"
pairing_scaling_seed_selection_metric = "compressed_test_mean_abs_error"
pairing_scaling_epoch_budgets = globals().get("PAIRING_SCALING_EPOCH_BUDGETS", [5000, 10000, 15000, 20000])

pairing_scaling_regime_row = (
    regime_summary.groupby("regime", as_index=False)[pairing_scaling_selection_metric]
    .mean()
    .sort_values(pairing_scaling_selection_metric)
    .iloc[0]
)
pairing_scaling_regime = pairing_scaling_regime_row["regime"]
pairing_scaling_n_values = [int(n) for n in pairing_sweep_settings["n_pmm_values"]]

pairing_scaling_seed_rows = {}
for n_pmm in pairing_scaling_n_values:
    candidates = sweep_detail[
        sweep_detail["regime"].eq(pairing_scaling_regime)
        & sweep_detail["n"].eq(n_pmm)
    ].copy()
    pairing_scaling_seed_rows[int(n_pmm)] = candidates.sort_values(pairing_scaling_seed_selection_metric).iloc[0]

best_interval = next(item for item in pairing_sweep_settings["intervals"] if item["name"] == pairing_scaling_regime)

scaling_settings = {
    "source": "best_pairing_config_from_regime_sweep",
    "selection_metric": pairing_scaling_selection_metric,
    "seed_selection_metric": pairing_scaling_seed_selection_metric,
    "best_regime": pairing_scaling_regime,
    "selected_sweep_epochs_by_n": {int(n): int(row["epochs"]) for n, row in pairing_scaling_seed_rows.items()},
    "selected_seed_by_n": {int(n): int(row["seed"]) for n, row in pairing_scaling_seed_rows.items()},
    "pnum": pairing_sweep_settings["pnum"],
    "hnum": pairing_sweep_settings["hnum"],
    "delta": pairing_sweep_settings["delta"],
    "g_min": float(best_interval["g_min"]),
    "g_max": float(best_interval["g_max"]),
    "n_train": int(best_interval["n_train"]),
    "n_test": int(best_interval["n_test"]),
    "k": pairing_sweep_settings["k"],
    "seed_by_n": {int(n): int(row["seed"]) for n, row in pairing_scaling_seed_rows.items()},
    "diagnostic_type": "multi_size_single_seed_epoch_scaling",
    "n_pmm_values": pairing_scaling_n_values,
    "epoch_budgets": pairing_scaling_epoch_budgets,
    "learning_rate": DEFAULT_LR,
    "init_scale": DEFAULT_INIT_SCALE,
}

scaling_data = make_pairing_data(
    pnum=scaling_settings["pnum"],
    hnum=scaling_settings["hnum"],
    delta=scaling_settings["delta"],
    gmin=scaling_settings["g_min"],
    gmax=scaling_settings["g_max"],
    n_train=scaling_settings["n_train"],
    n_test=scaling_settings["n_test"],
    k_levels=scaling_settings["k"],
)

scaling_rows = []
scaling_predictions = {}
print("Pairing scaling settings:", scaling_settings)
for n_pmm in scaling_settings["n_pmm_values"]:
    seed_for_n = int(scaling_settings["seed_by_n"][int(n_pmm)])
    for epochs in scaling_settings["epoch_budgets"]:
        print(f"Pairing scaling: regime={pairing_scaling_regime}, n={n_pmm}, seed={seed_for_n}, epochs={epochs}")
        case = train_pmm_on_data(
            scaling_data,
            n_pmm=n_pmm,
            k=scaling_settings["k"],
            seed=seed_for_n,
            epochs=epochs,
            lr=scaling_settings["learning_rate"],
            init_scale=scaling_settings["init_scale"],
        )
        scaling_rows.append({
            "regime": pairing_scaling_regime,
            "g_min": scaling_settings["g_min"],
            "g_max": scaling_settings["g_max"],
            "n": int(n_pmm),
            "k": int(scaling_settings["k"]),
            "seed": int(seed_for_n),
            "epochs": int(epochs),
            "test_mean_abs_error": case["test_mean_abs_error"],
            "test_max_abs_error": case["test_max_abs_error"],
            "test_rms_error": case["test_rms_error"],
            "test_loss": case["test_loss"],
            "train_loss": case["train_loss"],
            "train_time_s": case["train_time_s"],
            "best_step": case["best_step"],
        })
        scaling_predictions[(int(n_pmm), int(epochs))] = {
            "g_test": scaling_data["g_test"],
            "exact": scaling_data["E_test"],
            "pmm": case["pmm_pred"],
            "abs_error": np.abs(case["pmm_pred"] - scaling_data["E_test"]),
        }

scaling_detail = pd.DataFrame(scaling_rows).sort_values(["n", "epochs"]).reset_index(drop=True)
scaling_error_table = scaling_detail[["regime", "n", "epochs", "test_mean_abs_error", "test_max_abs_error", "test_rms_error", "test_loss"]]
scaling_train_time_table = scaling_detail[["regime", "n", "epochs", "train_time_s", "train_loss", "best_step"]]

pairing_results["pmm_size_epoch_scaling"] = {
    "settings": scaling_settings,
    "detail": scaling_detail,
    "error_table": scaling_error_table,
    "train_time_table": scaling_train_time_table,
    "predictions_by_key": scaling_predictions,
}
save_results(pairing_results)
scaling_detail


Pairing scaling settings: {'source': 'best_pairing_config_from_regime_sweep', 'selection_metric': 'mean_compressed_test_mean_abs_error', 'seed_selection_metric': 'compressed_test_mean_abs_error', 'best_regime': 'g0_2', 'selected_sweep_epochs_by_n': {16: 20000, 32: 20000, 48: 20000}, 'selected_seed_by_n': {16: 1236, 32: 1235, 48: 1236}, 'pnum': 10, 'hnum': 10, 'delta': 1.0, 'g_min': 0.0, 'g_max': 2.0, 'n_train': 40, 'n_test': 200, 'k': 3, 'seed_by_n': {16: 1236, 32: 1235, 48: 1236}, 'diagnostic_type': 'multi_size_single_seed_epoch_scaling', 'n_pmm_values': [16, 32, 48], 'epoch_budgets': [5000, 10000, 15000, 20000], 'learning_rate': 0.01, 'init_scale': 0.01}
Pairing scaling: regime=g0_2, n=16, seed=1236, epochs=5000
Pairing scaling: regime=g0_2, n=16, seed=1236, epochs=10000
Pairing scaling: regime=g0_2, n=16, seed=1236, epochs=15000
Pairing scaling: regime=g0_2, n=16, seed=1236, epochs=20000
Pairing scaling: regime=g0_2, n=32, seed=1235, epochs=5000
Pairing scaling: regime=g0_2, n=32, s

,regime,g_min,g_max,n,k,seed,epochs,test_mean_abs_error,test_max_abs_error,test_rms_error,test_loss,train_loss,train_time_s,best_step
0,g0_2,0.0,2.0,16,3,1236,5000,0.114766,0.437357,0.172939,2.990795e-02,3.037401e-02,3.442607,5000
1,g0_2,0.0,2.0,16,3,1236,10000,0.014448,0.066518,0.018132,3.287673e-04,3.639210e-04,6.667915,10000
2,g0_2,0.0,2.0,16,3,1236,15000,0.002086,0.014630,0.002841,8.068460e-06,9.269719e-06,9.799027,15000
3,g0_2,0.0,2.0,16,3,1236,20000,0.000742,0.004775,0.000967,9.345568e-07,1.058558e-06,13.036515,19850
4,g0_2,0.0,2.0,32,3,1235,5000,0.123473,0.511366,0.197895,3.916229e-02,3.925224e-02,11.861641,5000
5,g0_2,0.0,2.0,32,3,1235,10000,0.010980,0.050763,0.013708,1.879004e-04,2.090006e-04,23.321291,10000
6,g0_2,0.0,2.0,32,3,1235,15000,0.001785,0.009453,0.002231,4.977972e-06,5.652170e-06,34.145647,15000
7,g0_2,0.0,2.0,32,3,1235,20000,0.000730,0.003905,0.000950,9.017306e-07,9.791873e-07,43.181183,18750
8,g0_2,0.0,2.0,48,3,1236,5000,0.127126,0.544215,0.209671,4.396195e-02,4.383210e-02,27.464584,5000
9,g0_2,0.0,2.0,48,3,1236,10000,0.008733,0.042977,0.011026,1.215732e-04,1.358994e-04,53.724251,10000


## 3. Pairing Raw PMM vs Neural Network

Uses the compression-selected representative run from the pairing sweep, then compares the raw PMM predictions against a tuned feedforward neural-network baseline. The compressed PMM metrics are saved as metadata, but the main arrays/table in this section are raw PMM vs NN. The saved comparison table separates final training time from total preparation/selection time.


In [7]:
pmm_vs_nn_selection_metric = "compressed_test_mean_abs_error"
pmm_vs_nn_regime = "g0_2"
pmm_vs_nn_pmm_sizes = [int(n) for n in pairing_sweep_settings["n_pmm_values"]]

pmm_vs_nn_rows = []
pmm_vs_nn_runs = {}
for n_pmm in pmm_vs_nn_pmm_sizes:
    candidates = sweep_detail[
        sweep_detail["regime"].eq(pmm_vs_nn_regime)
        & sweep_detail["n"].eq(n_pmm)
    ].copy()
    selected_row = candidates.sort_values(pmm_vs_nn_selection_metric).iloc[0]
    selected_key = (selected_row["regime"], int(selected_row["n"]), int(selected_row["seed"]))
    pmm_vs_nn_rows.append(selected_row)
    pmm_vs_nn_runs[int(n_pmm)] = run_lookup[selected_key]

best_interval = next(item for item in pairing_sweep_settings["intervals"] if item["name"] == pmm_vs_nn_regime)

nn_settings = {
    "source": "g0_2_all_pmm_dimensions_raw_pmm_vs_nn",
    "regime": pmm_vs_nn_regime,
    "pnum": pairing_sweep_settings["pnum"],
    "hnum": pairing_sweep_settings["hnum"],
    "delta": pairing_sweep_settings["delta"],
    "g_min": float(best_interval["g_min"]),
    "g_max": float(best_interval["g_max"]),
    "n_train": int(best_interval["n_train"]),
    "n_test": int(best_interval["n_test"]),
    "k": pairing_sweep_settings["k"],
    "pmm_n_values": pmm_vs_nn_pmm_sizes,
    "pmm_selected_seed_by_n": {int(row["n"]): int(row["seed"]) for row in pmm_vs_nn_rows},
    "pmm_selected_epochs_by_n": {int(row["n"]): int(row["epochs"]) for row in pmm_vs_nn_rows},
    "pmm_selection_metric": pmm_vs_nn_selection_metric,
    "nn_widths": [32, 64, 128],
    "nn_activations": ["tanh", "relu"],
    "nn_weight_decays": [0.0, 1e-5],
    "nn_lr": 1e-3,
    "nn_epochs": NN_EPOCHS,
    "nn_batch_size": 16,
    "nn_seed": GLOBAL_SEED,
    "device": "cpu",
    "method_notes": {
        "representative_selection": "For the fixed g0_2 interval, each PMM dimension uses the best existing sweep seed selected by compressed PMM test mean absolute error. PMMs are reused from the sweep rather than retrained.",
        "timing": "PMM model_selection_time_s is the summed sweep train time for that PMM dimension and interval across seeds. NN model_selection_time_s is the summed grid-search train time.",
    },
}

nn_data = make_pairing_data(
    pnum=nn_settings["pnum"],
    hnum=nn_settings["hnum"],
    delta=nn_settings["delta"],
    gmin=nn_settings["g_min"],
    gmax=nn_settings["g_max"],
    n_train=nn_settings["n_train"],
    n_test=nn_settings["n_test"],
    k_levels=nn_settings["k"],
)
(
    X_train,
    y_train,
    X_val,
    y_val,
    train_idx,
    val_idx,
) = make_train_validation_split(nn_data["X_train"], nn_data["y_train"], val_fraction=0.25)

nn_grid_summary, best_nn = run_mlp_grid_search(
    X_train,
    y_train,
    X_val,
    y_val,
    widths=tuple(nn_settings["nn_widths"]),
    activations=tuple(nn_settings["nn_activations"]),
    weight_decays=tuple(nn_settings["nn_weight_decays"]),
    lr=nn_settings["nn_lr"],
    epochs=nn_settings["nn_epochs"],
    batch_size=nn_settings["nn_batch_size"],
    seed=nn_settings["nn_seed"],
    device=nn_settings["device"],
)

nn_final = train_final_mlp(
    nn_data["X_train"],
    nn_data["y_train"],
    hidden_width=best_nn["hidden_width"],
    activation=best_nn["activation"],
    weight_decay=best_nn["weight_decay"],
    lr=nn_settings["nn_lr"],
    epochs=nn_settings["nn_epochs"],
    batch_size=nn_settings["nn_batch_size"],
    seed=nn_settings["nn_seed"],
    device=nn_settings["device"],
)
nn_eval = evaluate_mlp_on_data(nn_final["model"], nn_data["X_test"], nn_data["y_test"], device=nn_settings["device"])
nn_pred = nn_eval["y_pred"]

nn_model_selection_time_s = float(nn_grid_summary["train_time"].sum())
nn_final_train_time_s = float(nn_final["train_time"])
nn_total_prep_time_s = nn_model_selection_time_s + nn_final_train_time_s

comparison_rows = []
pmm_metadata_by_n = {}
pmm_predictions_by_n = {}
compressed_pmm_predictions_by_n = {}
pmm_abs_error_by_n = {}
compressed_pmm_abs_error_by_n = {}

for row in pmm_vs_nn_rows:
    n_pmm = int(row["n"])
    pmm_run = pmm_vs_nn_runs[n_pmm]

    pmm_pred = pmm_run["pmm"]
    compressed_pred = pmm_run["compressed_pmm"]
    pmm_parameter_count_value = pmm_parameter_count(n_pmm, n_features=1)
    pmm_final_train_time_s = float(row["train_time_s"])
    pmm_model_selection_time_s = float(
        sweep_detail.loc[
            sweep_detail["regime"].eq(pmm_vs_nn_regime) & sweep_detail["n"].eq(n_pmm),
            "train_time_s",
        ].sum()
    )
    pmm_total_prep_time_s = pmm_model_selection_time_s

    stats = compute_error_stats(nn_data["E_test"], pmm_pred)
    comparison_rows.append({
        "model": f"PMM n={n_pmm}",
        "model_family": "PMM",
        "n": n_pmm,
        "seed": int(row["seed"]),
        "epochs": int(row["epochs"]),
        "mean_abs_test_error": stats["mean_abs_error"],
        "max_abs_test_error": stats["max_abs_error"],
        "rms_test_error": stats["rms_error"],
        "parameter_count": int(pmm_parameter_count_value),
        "final_train_time_s": pmm_final_train_time_s,
        "model_selection_time_s": pmm_model_selection_time_s,
        "total_prep_time_s": pmm_total_prep_time_s,
    })

    pmm_metadata_by_n[n_pmm] = {
        "regime": row["regime"],
        "n": n_pmm,
        "seed": int(row["seed"]),
        "epochs": int(row["epochs"]),
        "k": int(row["k"]),
        "selection_metric": pmm_vs_nn_selection_metric,
        "parameter_count": int(pmm_parameter_count_value),
        "train_time_s": pmm_final_train_time_s,
        "train_loss": float(row["train_loss"]),
        "raw_test_loss": float(row["test_loss"]),
        "raw_test_mean_abs_error": float(row["test_mean_abs_error"]),
        "raw_test_max_abs_error": float(row["test_max_abs_error"]),
        "raw_test_rms_error": float(row["test_rms_error"]),
        "compressed_test_loss": float(row["compressed_test_loss"]),
        "compressed_test_mean_abs_error": float(row["compressed_test_mean_abs_error"]),
        "compressed_test_max_abs_error": float(row["compressed_test_max_abs_error"]),
        "compressed_test_rms_error": float(row["compressed_test_rms_error"]),
        "model_selection_time_s": pmm_model_selection_time_s,
        "total_prep_time_s": pmm_total_prep_time_s,
    }
    pmm_predictions_by_n[n_pmm] = pmm_pred
    compressed_pmm_predictions_by_n[n_pmm] = compressed_pred
    pmm_abs_error_by_n[n_pmm] = np.abs(pmm_pred - nn_data["E_test"])
    compressed_pmm_abs_error_by_n[n_pmm] = np.abs(compressed_pred - nn_data["E_test"])

nn_stats = compute_error_stats(nn_data["E_test"], nn_pred)
comparison_rows.append({
    "model": "NN",
    "model_family": "NN",
    "n": np.nan,
    "seed": int(nn_settings["nn_seed"]),
    "epochs": int(nn_settings["nn_epochs"]),
    "mean_abs_test_error": nn_stats["mean_abs_error"],
    "max_abs_test_error": nn_stats["max_abs_error"],
    "rms_test_error": nn_stats["rms_error"],
    "parameter_count": int(nn_final["parameter_count"]),
    "final_train_time_s": nn_final_train_time_s,
    "model_selection_time_s": nn_model_selection_time_s,
    "total_prep_time_s": nn_total_prep_time_s,
})
comparison_table = pd.DataFrame(comparison_rows)

pairing_results["pmm_vs_nn"] = {
    "settings": nn_settings,
    "comparison_table": comparison_table,
    "pmm_metadata_by_n": pmm_metadata_by_n,
    "timing_notes": {
        "final_train_time_s": "Time for the selected/final model training run.",
        "pmm_selection_metric": "For each PMM dimension on g0_2, the representative seed is selected by compressed PMM quality; the main comparison table compares raw PMM predictions against NN predictions.",
        "model_selection_time_s": "For each PMM row, summed PMM sweep train time across seeds for that dimension and interval. For NN, summed grid-search train time.",
        "total_prep_time_s": "Model-selection time plus any separate final retraining time. The PMM selected runs are reused from the sweep, so there is no extra final retrain.",
    },
    "nn_grid_summary": nn_grid_summary,
    "nn_best_validation": {k: v for k, v in best_nn.items() if k not in {"model", "y_val_pred"}},
    "nn_final_summary": {k: v for k, v in nn_final.items() if k != "model"},
    "g_test": nn_data["g_test"],
    "exact": nn_data["E_test"],
    "pmm_by_n": pmm_predictions_by_n,
    "compressed_pmm_by_n": compressed_pmm_predictions_by_n,
    "nn": nn_pred,
    "pmm_abs_error_by_n": pmm_abs_error_by_n,
    "compressed_pmm_abs_error_by_n": compressed_pmm_abs_error_by_n,
    "nn_abs_error": np.abs(nn_pred - nn_data["E_test"]),
    "train_validation_indices": {"train_idx": train_idx, "val_idx": val_idx},
}
save_results(pairing_results)
comparison_table


Saved pairing results to results/pairing_results.pkl


,model,model_family,n,seed,epochs,mean_abs_test_error,max_abs_test_error,rms_test_error,parameter_count,final_train_time_s,model_selection_time_s,total_prep_time_s
0,PMM n=16,PMM,16.0,1236,20000,0.000742,0.004775,0.000967,512,12.617240,36.765815,36.765815
1,PMM n=32,PMM,32.0,1235,20000,0.000730,0.003905,0.000950,2048,43.380326,124.162764,124.162764
2,PMM n=48,PMM,48.0,1236,20000,0.000700,0.004096,0.000902,4608,105.298637,294.308830,294.308830
3,NN,NN,NaN,1234,2000,0.003631,0.031684,0.004852,17155,2.441306,17.813465,20.254771
